In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBRegressor

print("XGBoost is installed successfully!")

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import mean_squared_error, pairwise_distances, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [ ]:
# Load the datasets
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Check the dataset sizes
print("Training data:", train.shape)
print("Testing data:", test.shape)

In [ ]:
# Select the 9 molecular descriptors
features = [
    "molecular_weight",
    "logp",
    "hba",
    "hbd",
    "psa",
    "rtb",
    "ro5_violations",
    "aromatic_rings",
    "heavy_atoms"
]

# Select target values
y_train = train["pf_pEC50"]
y_test = test["pf_pEC50"]

# Select molecular descriptors
X_train = train[features]
X_test = test[features]

print("Training descriptors:", X_train.shape)
print("Testing descriptors:", X_test.shape)

In [ ]:
# Create Morgan fingerprint generator
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024
)

# Function to convert SMILES into fingerprints
def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return generator.GetFingerprintAsNumPy(mol)

# Generate fingerprints
train_fps = [smiles_to_fingerprint(s) for s in train["smiles"]]
test_fps = [smiles_to_fingerprint(s) for s in test["smiles"]]

# Check for invalid structures
print("Invalid training structures:", sum(fp is None for fp in train_fps))
print("Invalid testing structures:", sum(fp is None for fp in test_fps))

In [ ]:
# Convert fingerprints into NumPy arrays
X_train_fp = np.array(train_fps)
X_test_fp = np.array(test_fps)

# Combine fingerprints with molecular descriptors
X_train_combined = np.hstack((
    X_train_fp,
    X_train.to_numpy()
))

X_test_combined = np.hstack((
    X_test_fp,
    X_test.to_numpy()
))

print("Combined training shape:", X_train_combined.shape)
print("Combined testing shape:", X_test_combined.shape)

In [ ]:
# Create XGBoost regression model
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Evaluate using 5-fold cross-validation
xgb_cv_scores = cross_val_score(
    xgb_model,
    X_train_combined,
    y_train,
    cv=5,
    scoring="r2"
)

print("XGBoost CV R² scores:", xgb_cv_scores.round(4))
print("Mean XGBoost CV R²:", round(xgb_cv_scores.mean(), 4))

## Tanimoto Similarity-Based SVR

This approach uses Tanimoto similarity between Morgan fingerprints to predict antiparasitic activity.

Unlike standard SVR, this model directly compares molecular fingerprints based on their shared structural features.

Performance is evaluated using five-fold cross-validation on the training dataset.

In [ ]:
# Convert fingerprints into Boolean arrays
X_train_binary = X_train_fp.astype(bool)

# Calculate Tanimoto similarity between training compounds
# For binary fingerprints, Tanimoto = 1 - Jaccard distance
tanimoto_train = 1 - pairwise_distances(
    X_train_binary,
    metric="jaccard",
    n_jobs=-1
)

print("Tanimoto similarity matrix:", tanimoto_train.shape)

In [ ]:
# Create cross-validation splits
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Create Tanimoto SVR
tanimoto_svr = SVR(kernel="precomputed")

# Test different regularisation settings
param_grid = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.1, 0.2]
}

# Tune using training data only
tanimoto_grid = GridSearchCV(
    tanimoto_svr,
    param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

tanimoto_grid.fit(tanimoto_train, y_train)

print("Best parameters:", tanimoto_grid.best_params_)
print("Best Tanimoto CV R²:", round(tanimoto_grid.best_score_, 4))

In [ ]:
# Same unshuffled folds used for earlier models
cv = KFold(n_splits=5, shuffle=False)

# Use the best Tanimoto SVR settings
tanimoto_final = SVR(
    kernel="precomputed",
    C=1,
    epsilon=0.1
)

# Evaluate using identical folds
tanimoto_scores = cross_val_score(
    tanimoto_final,
    tanimoto_train,
    y_train,
    cv=cv,
    scoring="r2"
)

print("Tanimoto CV R² scores:", tanimoto_scores.round(4))
print("Mean Tanimoto CV R²:", round(tanimoto_scores.mean(), 4))

In [ ]:
# Use the best SVR model from GridSearchCV
best_svr = svr_grid.best_estimator_

# Predict the test compounds
svr_test_predictions = best_svr.predict(X_test_combined)

# Calculate test performance
svr_test_r2 = r2_score(y_test, svr_test_predictions)
svr_test_rmse = np.sqrt(mean_squared_error(y_test, svr_test_predictions))

print("Final SVR Results")
print("-----------------")
print(f"Test R²: {svr_test_r2:.4f}")
print(f"Test RMSE: {svr_test_rmse:.4f}")

In [ ]:
best_svr = svr_grid.best_estimator_

svr_test_predictions = best_svr.predict(X_test_combined)

svr_test_r2 = r2_score(y_test, svr_test_predictions)
svr_test_rmse = np.sqrt(mean_squared_error(y_test, svr_test_predictions))

print("Final SVR Results")
print("-----------------")
print(f"Test R²: {svr_test_r2:.4f}")
print(f"Test RMSE: {svr_test_rmse:.4f}")

In [ ]:
# Use best XGBoost hyperparameters
best_xgb = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

best_xgb.fit(X_train_combined, y_train)

# Predict test values
xgb_test_predictions = best_xgb.predict(X_test_combined)

print("Final XGBoost Results")
print("---------------------")
print(f"Test R²: {r2_score(y_test, xgb_test_predictions):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, xgb_test_predictions)):.4f}")